In [1]:
import os
import time
import math
import torch
import torch.nn as nn
os.chdir('..')
from models.transformer import Transformer
from transformers import AutoTokenizer

/home/hussin/miniconda3/envs/caduceus_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
def get_vram_mb():
    return torch.cuda.max_memory_allocated() / (1024 ** 2) if torch.cuda.is_available() else 0.0

In [3]:
device_name = torch.cuda.get_device_name(0)

In [4]:
device_name

'NVIDIA GeForce RTX 2080 Ti'

In [5]:
def detect_gpu_name():
    """Detects connected CUDA device name."""
    if not torch.cuda.is_available():
        return "CPU"

    return torch.cuda.get_device_name(0)


In [6]:
detect_gpu_name()

'NVIDIA GeForce RTX 2080 Ti'

In [7]:
GPU_PEAK_BANDWIDTH_GBS = { 
"NVIDIA GeForce RTX 2080 Ti": 448 
}

In [8]:
import torch
import torch.nn as nn

def generate_with_metrics(
    model,
    input_ids,
    max_new_tokens=128,
    temperature=0.7,
    top_k=50,
):
    model.eval()
    device = input_ids.device

    # 1. Dynamic Parameter Byte Calculation
    weight_bytes = sum(p.numel() * p.element_size() for p in model.parameters())
    _, prompt_len = input_ids.shape

    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.synchronize()

    # Practice 1: Warm-up pass to ensure CUDA context / compile graphs are initialized
    with torch.no_grad():
        _ = model(input_ids)
    torch.cuda.synchronize()

    start_event = torch.cuda.Event(enable_timing=True)
    prefill_event = torch.cuda.Event(enable_timing=True)

    # --- PREFILL PHASE (first token) ---
    start_event.record()
    with torch.amp.autocast(device_type=device.type, dtype=torch.float16):
        logits = model(input_ids)
        next_token_logits = logits[:, -1, :] / temperature

        if top_k is not None:
            v, _ = torch.topk(next_token_logits, min(top_k, next_token_logits.size(-1)))
            next_token_logits[next_token_logits < v[:, [-1]]] = -float("Inf")

        probs = nn.functional.softmax(next_token_logits, dim=-1)
        next_token = torch.multinomial(probs, num_samples=1)

    prefill_event.record()
    curr_input_ids = torch.cat([input_ids, next_token], dim=1)

    # --- DECODE PHASE (single token step) ---
    total_bytes_moved_decode = 0
    step_times_ms = []

    if max_new_tokens > 1:
        total_bytes_moved_decode = weight_bytes

        step_start = torch.cuda.Event(enable_timing=True)
        step_end = torch.cuda.Event(enable_timing=True)

        # Practice 2: Record timing strictly around the model decode forward pass
        step_start.record()
        with torch.amp.autocast(device_type=device.type, dtype=torch.float16):
            logits = model(curr_input_ids)
            next_token_logits = logits[:, -1, :] / temperature

            if top_k is not None:
                v, _ = torch.topk(next_token_logits, min(top_k, next_token_logits.size(-1)))
                next_token_logits[next_token_logits < v[:, [-1]]] = -float("Inf")

            probs = nn.functional.softmax(next_token_logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
        step_end.record()

        torch.cuda.synchronize()
        step_duration_ms = step_start.elapsed_time(step_end)
        step_times_ms.append(step_duration_ms)

        curr_input_ids = torch.cat([curr_input_ids, next_token], dim=1)

    # Practice 3: Calculate TTFT, TPOT, and MBU using isolated timings
    ttft_ms = start_event.elapsed_time(prefill_event)
    avg_tpot_ms = step_times_ms[0] if step_times_ms else 0.0

    decode_seconds = avg_tpot_ms / 1000.0
    achieved_bandwidth_gbs = (total_bytes_moved_decode / 1e9) / decode_seconds if decode_seconds > 0 else 0.0
    throughput_tok_sec = (1.0 / decode_seconds) if decode_seconds > 0 else 0.0

    peak_bw = 448.0  # Peak bandwidth for RTX 2080 GDDR6 in GB/s
    mbu_percent = (achieved_bandwidth_gbs / peak_bw) * 100.0 if peak_bw > 0 else 0.0

    generated_tokens = curr_input_ids[0, prompt_len:].tolist()

    return {
        "generated_tokens": len(generated_tokens),
        "ttft_ms": round(ttft_ms, 2),
        "tpot_ms": round(avg_tpot_ms, 2),
        "throughput_tok_sec": round(throughput_tok_sec, 2),
        "total_gb_moved": round(total_bytes_moved_decode / 1e9, 4),
        "achieved_bandwidth_gbs": round(achieved_bandwidth_gbs, 2),
        "mbu_percent": round(mbu_percent, 2),
        "peak_vram_mb": round(torch.cuda.max_memory_allocated() / (1024 ** 2), 2)
    }, generated_tokens

### GEMV (sequence length of 1) for MHA basline model with torch.compile(mode = "reduce_overhead")

In [9]:
checkpoint_path = "checkpoints/1_baseline_mha_sequential_res/model.pt"
checkpoint = torch.load(checkpoint_path, map_location=device)

model = Transformer(**checkpoint["model_args"]).to(device)
model.load_state_dict(checkpoint["model_state_dict"])

tokenizer = AutoTokenizer.from_pretrained("gpt2")

# 1. Compile ONCE outside the benchmarking function
print("Compiling model...")
compiled_model = torch.compile(model, mode="reduce-overhead", fullgraph=True)


# Wait for GPU compilation to complete completely
torch.cuda.synchronize()
print("Compilation complete. Starting benchmark...")

Compiling model...
Compilation complete. Starting benchmark...


In [10]:
# 3. Target Input Preparation
prompt = "Once upon a time in a deep forest, there lived a small"
input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)

print(f"Prompt Length: {input_ids.shape[1]} tokens")

# 4. WARMUP PHASE (Triggers compilation & CUDA Graph capture for shapes [1, N] and [1, N+1])
print("Warming up CUDA context and capturing graphs...")
for _ in range(3):
    _ = generate_with_metrics(compiled_model, input_ids, max_new_tokens=2)

torch.cuda.synchronize()
print("Warmup complete. Starting benchmark run...")

# 5. BENCHMARK RUN (Zero compilation overhead, pure steady-state execution)
metrics, generated_tokens = generate_with_metrics(
    compiled_model, 
    input_ids, 
    max_new_tokens=2, 
    temperature=0.7, 
    top_k=50
)

# 6. Decode and Display Output
generated_text = tokenizer.decode(generated_tokens, skip_special_tokens=True)

print("\n================ BENCHMARK RESULTS ================")
print(f"Generated Text:         '{generated_text}'")
print(f"TTFT (Prefill):         {metrics['ttft_ms']} ms")
print(f"TPOT (Single Decode):   {metrics['tpot_ms']} ms")
print(f"Throughput:             {metrics['throughput_tok_sec']} tok/sec")
print(f"Achieved Bandwidth:     {metrics['achieved_bandwidth_gbs']} GB/s")
print(f"MBU Utilization:        {metrics['mbu_percent']}%")
print(f"Peak VRAM Allocated:    {metrics['peak_vram_mb']} MB")
print("===================================================")

Prompt Length: 13 tokens
Warming up CUDA context and capturing graphs...


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The 

Warmup complete. Starting benchmark run...

================ BENCHMARK RESULTS ================
Generated Text:         ' house.'
TTFT (Prefill):         1.52 ms
TPOT (Single Decode):   1.48 ms
Throughput:             675.91 tok/sec
Achieved Bandwidth:     142.64 GB/s
MBU Utilization:        31.84%
Peak VRAM Allocated:    407.62 MB


### Profiling to monitor GPU utilization

In [13]:
import torch

# 1. Explain graph breaks in your model
explanation = torch._dynamo.explain(compiled_model)(input_ids)
print(explanation)

Graph Count: 1
Graph Break Count: 0
Op Count: 226
Break Reasons:
Ops per Graph:
  Ops 1:
    <built-in function add>
    <built-in method rsqrt of type object at 0x7f1f83e85840>
    <built-in function mul>
    <built-in function mul>
    <built-in function getitem>
    <built-in function getitem>
    <built-in function mul>
    <built-in function getitem>
    <built-in function getitem>
    <built-in function neg>
    <built-in method cat of type object at 0x7f1f83e85840>
    <built-in function mul>
    <built-in function add>
    <built-in function mul>
    <built-in function getitem>
    <built-in function getitem>
    <built-in function neg>
    <built-in method cat of type object at 0x7f1f83e85840>
    <built-in function mul>
    <built-in function add>
    <built-in method matmul of type object at 0x7f1f83e85840>
    <built-in function truediv>
    <built-in method ones of type object at 0x7f1f83e85840>
    <built-in method tril of type object at 0x7f1f83e85840>
    <built-in func

In [16]:
import torch

# 1. Warm up raw compiled model pass
with torch.no_grad():
    for _ in range(10):
        _ = compiled_model(input_ids)
torch.cuda.synchronize()

# 2. Measure ONLY the raw model forward pass (Same as gpu_profile.py)
start_event = torch.cuda.Event(enable_timing=True)
end_event = torch.cuda.Event(enable_timing=True)

start_event.record()
with torch.no_grad():
    for _ in range(100):
        _ = compiled_model(input_ids)
end_event.record()
torch.cuda.synchronize()

raw_model_ms = start_event.elapsed_time(end_event) / 100.0

print(f"Isolated Model Latency in Notebook: {raw_model_ms:.2f} ms")
print(f"bandwidth for this run is {metrics['total_gb_moved'] / (raw_model_ms / 1000.0):.2f} GB/s")
print(f"MBU Utilization for this run is {(metrics['total_gb_moved'] / (raw_model_ms / 1000.0) )  / GPU_PEAK_BANDWIDTH_GBS[device_name] * 100:.2f}%")

Isolated Model Latency in Notebook: 0.85 ms
bandwidth for this run is 248.21 GB/s
MBU Utilization for this run is 55.40%


## Single-Token Decode Benchmarking & Memory Bandwidth Utilization (MBU)

### 1. Performance Evolution Matrix

| Execution Mode | Isolated Op / Phase | Latency (ms) | Effective Bandwidth | Achieved MBU (%) |
| :--- | :--- | :--- | :--- | :--- |
| **Eager PyTorch** | Full Forward Pass | $3.79\text{ ms}$ | $51.86\text{ GB/s}$ | $11.58\%$ |
| **Notebook (`torch.compile`)** | Full Generation (Model + Sampling) | $1.48\text{ ms}$ | $142.72\text{ GB/s}$ | $31.86\%$ |
| **Notebook (`torch.compile`)** | **Isolated Model Forward Pass** | **$0.85\text{ ms}$** | **$248.21\text{ GB/s}$** | **$55.40\%$** |

---

### 2. Key Diagnostic Findings

1. **Compilation Speedup**:
   * Enabling `torch.compile(mode="reduce-overhead")` reduces isolated model forward pass latency from **$3.79\text{ ms}$ down to $0.85\text{ ms}$** inside the notebook (a **$4.46\times$ speedup**).
   * Memory Bandwidth Utilization (MBU) increases from **$11.58\%$ to $55.40\%$**, confirming that CUDA Graph capture successfully eliminated host CPU driver launch gaps.

2. **Quantifying Sampling Overhead**:
   * **Full Generation Step** (Model + `top_k` + `softmax` + `multinomial` + `autocast`): **$1.48\text{ ms}$**
   * **Isolated Model Step**: **$0.85\text{ ms}$**
   * **Inference**: Non-fused auxiliary sampling operations account for **$0.63\text{ ms}$** ($\sim 42.5\%$ of total single-token decode latency).



### GEMV (sequence length of 1) for GQA basline model with torch.compile("reduce_overhead")

In [ ]:
import gc
del model
gc.collect()
torch.cuda.empty_cache()


In [ ]:
checkpoint_path = "checkpoints/4_variant_c_gqa_parallel_res/model.pt"

checkpoint = torch.load(checkpoint_path, map_location=device)

model = Transformer(**checkpoint["model_args"]).to(device)
model.load_state_dict(checkpoint["model_state_dict"])

tokenizer = AutoTokenizer.from_pretrained("gpt2")
# 1. Compile ONCE outside the benchmarking function
print("Compiling model...")
compiled_model = torch.compile(model, mode="reduce-overhead" , fullgraph=True)


# Wait for GPU compilation to complete completely
torch.cuda.synchronize()
print("Compilation complete. Starting benchmark...")

In [ ]:
# 3. Target Input Preparation
prompt = "Once upon a time in a deep forest, there lived a small"
input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)

print(f"Prompt Length: {input_ids.shape[1]} tokens")

# 4. WARMUP PHASE (Triggers compilation & CUDA Graph capture for shapes [1, N] and [1, N+1])
print("Warming up CUDA context and capturing graphs...")
for _ in range(3):
    _ = generate_with_metrics(compiled_model, input_ids, max_new_tokens=2)

torch.cuda.synchronize()
print("Warmup complete. Starting benchmark run...")

# 5. BENCHMARK RUN (Zero compilation overhead, pure steady-state execution)
metrics, generated_tokens = generate_with_metrics(
    compiled_model, 
    input_ids, 
    max_new_tokens=2, 
    temperature=0.7, 
    top_k=50
)

# 6. Decode and Display Output
generated_text = tokenizer.decode(generated_tokens, skip_special_tokens=True)

print("\n================ BENCHMARK RESULTS ================")
print(f"Generated Text:         '{generated_text}'")
print(f"TTFT (Prefill):         {metrics['ttft_ms']} ms")
print(f"TPOT (Single Decode):   {metrics['tpot_ms']} ms")
print(f"Throughput:             {metrics['throughput_tok_sec']} tok/sec")
print(f"Achieved Bandwidth:     {metrics['achieved_bandwidth_gbs']} GB/s")
print(f"MBU Utilization:        {metrics['mbu_percent']}%")
print(f"Peak VRAM Allocated:    {metrics['peak_vram_mb']} MB")
print("===================================================")

In [ ]:
print(metrics["total_gb_moved"])